In [1]:
import math
import numpy as np
import tensorflow as tf

from plotly.subplots import make_subplots
import plotly.graph_objects as go

import ipywidgets

from optical_communication_system.components import *
from optical_communication_system.datatypes import ElectricalSignal, OpticalSignal, Timing
from optical_communication_system.utils import dbm_to_w, from_db
from optical_communication_system.plot import *

2024-11-04 18:27:25.264955: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2024-11-04 18:27:25.301997: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-11-04 18:27:25.302024: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-11-04 18:27:25.303210: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-04 18:27:25.309190: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2024-11-04 18:27:25.309666: I tensorflow/core/platform/cpu_feature_guard.cc:1

In [2]:
Component.trace_debug_enabled_by_default = False

sampling_rate = 200e9
central_frequency = 193.414e12 # Hz

system = Composition([
    GaussianPulseGenerator(fwhm_duration=1e-9, power=2e-3, batch_size=1, name="pulse_gen"),
    OpticalFiber(
        attenuation=0.22,
        length=40.0,
        chromatic_dispersion=17.0,
        eff_non_linear_coeff=1.76,
        central_frequency=central_frequency,
        step_count=1000,
        name="fiber"
    )
])

In [6]:
def plot(attenuation, length, chromatic_dispersion, eff_non_linear_coeff, fwhm_duration_ps, power_dbm, sampling_rate_e9, sample_count):
    parameters = dict(
        fiber__attenuation=attenuation,
        fiber__length=length,
        fiber__chromatic_dispersion=chromatic_dispersion,
        fiber__eff_non_linear_coeff=eff_non_linear_coeff,
        pulse_gen__fwhm_duration=fwhm_duration_ps * 1e-12,
        pulse_gen__power=dbm_to_w(power_dbm)
    )
    timing = Timing(sampling_rate=sampling_rate_e9 * 1e9, sample_count=sample_count)
    
    system.trace(timing, **parameters)

    fig = plot_power(system.find_child("pulse_gen").output, label="Pulse")
    plot_power(system.output, fig, label="Pulse after fiber")

    fig.update_layout(autosize=True, height=600, margin=dict(t=30, b=30))

    return fig

ipywidgets.interact(
    plot,
    attenuation=(0.0, 0.4, 0.01),
    length=(0.0, 1000.0, 1.0),
    chromatic_dispersion=(-500.0, 500.0, 1.0),
    eff_non_linear_coeff=(0.0, 1e3, 1.0),
    fwhm_duration_ps=(1.0, 500.0, 1.0),
    power_dbm=(-15.0, 17.0, 0.5),
    sampling_rate_e9=(10.0, 1000.0, 1.0),
    sample_count=(100, 3000, 10)
)

interactive(children=(FloatSlider(value=0.2, description='attenuation', max=0.4, step=0.01), FloatSlider(value…

<function __main__.plot(attenuation, length, chromatic_dispersion, eff_non_linear_coeff, fwhm_duration_ps, power_dbm, sampling_rate_e9, sample_count)>